# Spectral heatmap animations of a 2×3 system (archived)

Heatmaps of the $x_1$ and $y_2$ position PSDs, the heterodyne and the homodyne spectrum of a
2×3 system (two beads with x, y, z), over frequency and one parameter, animated over a second
parameter: tweezer phase $\phi_2$ vs. coupling $g$, $g$ vs. $\phi_2$, detuning Δ vs. $g$, and $g$ vs.
air pressure. Each section is a pair of cells: compute the frames, then write the animation.

**Runtime:** about a minute with `MAKE_ANIMATIONS = False` (10 kHz grid, at most 3 values per
parameter, so each figure is a static preview showing the first frame). With
`MAKE_ANIMATIONS = True` (1 kHz grid, up to 49 frames × 40 rows per animation, 16 animations
written to `output/`) expect many hours.

**Updated Nov 2026** to current levisim:

- `calculate_all_hybridised_position_spectra(method)` → `all_position_power_spectra(method,
  zpf_scaled=False)[0]` (m²/Hz, as before), and the other `calculate_*` renames;
- the toy parameter `'φ2'` (phase of the 2nd tweezer) → `'φ_even_n'`, its name since Dec 2025;
  the old key is ignored by today's `MultiTrap`, so without the rename the phase animations would
  silently show no phase dependence;
- the 2024 `'model': 'pairs'` keys `first_Ωx`/`Ω_gap` are written as `Ω_first`,
  `Ω_interpair_gap` and `Ω_intrapair_gap = Ω_gap/10` (the same frequencies);
- animations are written to `output/` by `save_animation` instead of embedded as HTML.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.animation import writers

from levisim.core.plotting import set_plt_rcParams_to_xmgrace_style

# MAKE_ANIMATIONS = True computes every frame at full resolution and writes the animations to
# output/ (mp4 with ffmpeg, else gif). False (the smoke-test setting) draws a quick static preview
MAKE_ANIMATIONS = False
OUTPUT_DIR = Path('output')

set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0


def save_animation(animation, name):
    """Writes `animation` to output/<name>.mp4 (or .gif without ffmpeg), never into the notebook"""
    OUTPUT_DIR.mkdir(exist_ok=True)
    writer = 'ffmpeg' if writers.is_available('ffmpeg') else 'pillow'
    path = OUTPUT_DIR / f'{name}.{"mp4" if writer == "ffmpeg" else "gif"}'
    animation.save(path, writer=writer)
    print(f'Wrote {path}')
    return path

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.animation import ArtistAnimation
from tqdm import tqdm
from levisim import *

num_traps, Cartesian_axes_per_trap = 2, 3

toy_params = {
    # the 2024 keys 'model': 'pairs', first_Ωx = 312 kHz and Ω_gap = 15 kHz, in today's names
    'Ω_model': 'pairs',
    'Ω_first': 312*kHz,
    'Ω_interpair_gap': 15*kHz,
    'Ω_intrapair_gap': 1.5*kHz,
    'Ωz_to_Ωx_multiplier': 10,

    'toy_gx': -40*kHz,
    'toy_gy': -40*kHz,
    # phase of the even-numbered tweezers (called 'φ2' before Dec 2025)
    'φ_even_n': 0*degrees,

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'κ': 400*kHz,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-8*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}

# use 0-5 for psds to index the array given by all_position_power_spectra()[0]
spec_choice_to_str = {
    0: '$S_{x_1 x_1}$',
    1: '$S_{y_1 y_1}$',
    2: '$S_{z_1 z_1}$',
    3: '$S_{x_2 x_2}$',
    4: '$S_{y_2 y_2}$',
    5: '$S_{z_2 z_2}$',
    'het': '$S_{het}$',
    'hom': '$S_{hom}$',
}
spec_choice_to_units = {
    0: '(m$^2$/Hz)',
    1: '(m$^2$/Hz)',
    2: '(m$^2$/Hz)',
    3: '(m$^2$/Hz)',
    4: '(m$^2$/Hz)',
    5: '(m$^2$/Hz)',
    'het': '(photons/Hz)',
    'hom': '(photons/Hz)',
}

ω_range = np.arange(-500, 500, 1 if MAKE_ANIMATIONS else 10) * kHz
x = ω_range / kHz

class Parameter:
    def __init__(
        self,
        name,
        symbol,

        units,
        unit_symbol,
        unit_string,

        default_value,
        low, high, num_steps,
        log=False,
    ):
        self.name = name
        self.symbol = symbol

        self.units = units
        self.unit_symbol = unit_symbol
        self.unit_string = unit_string

        self.ylabel = f'{self.symbol} ({self.unit_string})'

        self.default_value = default_value

        if not MAKE_ANIMATIONS:
            # quick preview: at most 3 values per parameter (start, middle, end)
            num_steps = min(num_steps, 3)
        range_func = np.logspace if log else np.linspace
        self.range = range_func(low, high, num_steps) * units
        self.axis_range = self.range / self.units

phase = Parameter(
    name='phase',
    symbol=r'$\phi_2$',
    units=degrees,
    unit_symbol='°',
    unit_string='degrees',
    default_value=toy_params['φ_even_n'],
    low=0, high=360, num_steps=48 + 1,
)
coupling = Parameter(
    name='coupling',
    symbol='$g$',
    units=kHz,
    unit_symbol=' kHz',
    unit_string='kHz',
    default_value=toy_params['toy_gx'],
    # ignore g = 0 kHz case
    low=-5, high=-200, num_steps=40,
)
detuning = Parameter(
    name='detuning',
    symbol=r'$\Delta$',
    units=kHz,
    unit_symbol=' kHz',
    unit_string='kHz',
    default_value=cavity_kwargs['Δ'],
    low=0, high=-800, num_steps=16 + 1,
)
damping = Parameter(
    name='damping',
    symbol=r'$\kappa$',
    units=kHz,
    unit_symbol=' kHz',
    unit_string='kHz',
    default_value=cavity_kwargs['κ'],
    low=0, high=500, num_steps=10 + 1,
)
pressure = Parameter(
    name='pressure',
    symbol='$P_{air}$',
    units=mbar,
    unit_symbol=' mbar',
    unit_string='mbar',
    default_value=gas_kwargs['P'],
    low=1, high=-12, num_steps=13 + 1,
    log=True,
)
all_params = [phase, coupling, detuning, damping, pressure]

def calculate_intensity(
    spec_choice,
    y,
    frame_name,
    frame_value,
    method=QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION,
):
    y_variables = y.range
    y_var_name = y.name
    intensity = np.zeros(shape=(len(ω_range), len(y.range)))

    if frame_name == 'coupling':
        toy_params['toy_gx'] = frame_value
        toy_params['toy_gy'] = frame_value
    elif frame_name == 'phase':
        toy_params['φ_even_n'] = frame_value
    elif frame_name == 'detuning':
        cavity_kwargs['Δ'] = frame_value
    elif frame_name == 'damping':
        cavity_kwargs['κ'] = frame_value
    elif frame_name == 'pressure':
        gas_kwargs['P'] = frame_value

    for ii, y_value in enumerate(y.range):
        if y.name == 'coupling':
            toy_params['toy_gx'] = y_value
            toy_params['toy_gy'] = y_value
        elif y.name == 'phase':
            toy_params['φ_even_n'] = y_value
        elif y.name == 'detuning':
            cavity_kwargs['Δ'] = y_value
        elif y.name == 'damping':
            cavity_kwargs['κ'] = y_value
        elif y.name == 'pressure':
            gas_kwargs['P'] = y_value

        mtr = MultiTrap(
            num_traps=num_traps,
            Cartesian_axes_per_trap=Cartesian_axes_per_trap,
            ω_range=ω_range,

            cavity_kwargs=cavity_kwargs,
            gas_kwargs=gas_kwargs,
            toy_params=toy_params,
        )

        if spec_choice == 'het':
            intensity[:, ii] = mtr.heterodyne_spectrum(method)
        elif spec_choice == 'hom':
            intensity[:, ii] = mtr.homodyne_spectrum(method)
        else:
            intensity[:, ii] = mtr.all_position_power_spectra(method, zpf_scaled=False)[0][:, spec_choice]

    return intensity.T

def generate_heatmap_animation(
    spec_choice,
    x, y, frame,
    other_param_1, other_param_2, other_param_3,
):
    frames = []
    intensities = []
    global_min = float('inf')
    global_max = float('-inf')
    for frame_var_value in tqdm(frame.range, desc='global min/max calculation'):
        # x is always frequency
        intensity = calculate_intensity(
            spec_choice,
            y,
            frame.name,
            frame_var_value,
        )
        intensities.append(intensity)
        global_min = min(global_min, intensity.min())
        global_max = max(global_max, intensity.max())

    if spec_choice == 'hom':
        squeezed_threshold = 1
        norm_positive = mcolors.LogNorm(vmin=squeezed_threshold, vmax=global_max)
    else:
        norm = mcolors.LogNorm(vmin=global_min, vmax=global_max)

    fig, ax = plt.subplots(figsize=(7, 7))
    for ii, frame_var_value in enumerate(frame.range):

        # first, generate a title that updates each frame
        fs = [
            '.0e' if param.name == 'pressure' else '.0f'
            for param in [frame, other_param_1, other_param_2, other_param_3]
        ]
        ax.set_xlabel('Frequency (kHz)')
        ax.set_ylabel(y.ylabel)
        title_str = (
            f'{num_traps}×{Cartesian_axes_per_trap} system'
            f'\n'
            f'{spec_choice_to_str[spec_choice]}'
            f'(ω, '
            f'{y.symbol}'
            f')'
            f' at '
            f'{frame.symbol}'
            f'='
            f'{frame_var_value / frame.units:{fs[0]}}{frame.unit_symbol}'
            f'\n'
            f'({other_param_1.symbol}, {other_param_2.symbol}, {other_param_3.symbol})'
            f'='
            f'('
            f'{other_param_1.default_value / other_param_1.units:{fs[1]}}{other_param_1.unit_symbol}'
            f', '
            f'{other_param_2.default_value / other_param_2.units:{fs[2]}}{other_param_2.unit_symbol}'
            f', '
            f'{other_param_3.default_value / other_param_3.units:{fs[3]}}{other_param_3.unit_symbol}'
            f')'
        )
        title = ax.text(
            # position above the plot
            0.5, 1.03,
            title_str,
            size='small',
            ha='center',
            transform=ax.transAxes,
        )

        # next, generate the heatmap frames
        if spec_choice == 'hom':
            S_hom = intensities[ii]

            # split into squeezed and non-squeezed
            non_squeezed = np.where(S_hom >= squeezed_threshold, S_hom, np.nan)
            squeezed = np.where(S_hom < squeezed_threshold, S_hom, np.nan)

            # plot non-squeezed values on a log scale
            pc_positive = ax.pcolormesh(
                x,
                y.axis_range,
                non_squeezed,
                cmap='Wistia_r',
                norm=norm_positive,
                shading='nearest',
            )
            # plot squeezed values on a linear scale
            pc_negative = ax.pcolormesh(
                x,
                y.axis_range,
                squeezed,
                cmap='coolwarm',
                vmin=global_min,
                vmax=squeezed_threshold,
                shading='nearest',
            )
            fig.subplots_adjust(bottom=0.2)
            frames.append([pc_positive, pc_negative, title])

        else:
            pc_mesh = ax.pcolormesh(
                x,
                y.axis_range,
                intensities[ii],
                shading='nearest',
                norm=norm,
                cmap='plasma' if spec_choice == 'het' else 'viridis',
            )
            frames.append([pc_mesh, title])

    # lastly, add legend/colourbar outside of frame loop
    if spec_choice == 'hom':
        # [left, bottom, width, height]
        cax_positive = fig.add_axes([0.1, 0.05, 0.35, 0.03])
        cbar_positive = plt.colorbar(pc_positive, cax=cax_positive, orientation='horizontal')
        cbar_positive.ax.xaxis.set_label_position('top')
        cbar_positive.set_label('Non-Squeezed $S_{hom} ≥ 1$')

        cax_negative = fig.add_axes([0.5, 0.05, 0.35, 0.03])
        cbar_negative = plt.colorbar(pc_negative, cax=cax_negative, orientation='horizontal')
        cbar_negative.ax.xaxis.set_label_position('top')
        cbar_negative.set_label('Squeezed $S_{hom} < 1$')
    else:
        cbar = fig.colorbar(pc_mesh, ax=ax, location='right', pad=0.1)
        cbar.set_label(f'{spec_choice_to_str[spec_choice]} {spec_choice_to_units[spec_choice]}')

    return ArtistAnimation(fig, frames, blit=False)


# PSDs $\phi_2$ vs. $g$

In [ ]:
spec_choice = 0
ani_params = (
    spec_choice,
    x,

    phase,
    coupling,

    detuning,
    damping,
    pressure,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'phase vs coupling x1 PSD')

In [ ]:
spec_choice = 4
ani_params = (
    spec_choice,
    x,

    phase,
    coupling,

    detuning,
    damping,
    pressure,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'phase vs coupling y2 PSD')

# Het. $\phi_2$ vs. $g$

In [ ]:
spec_choice = 'het'
ani_params = (
    spec_choice,
    x,

    phase,
    coupling,

    detuning,
    damping,
    pressure,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'phase vs coupling heterodyne')

# Hom. $\phi_2$ vs. $g$

In [ ]:
spec_choice = 'hom'
ani_params = (
    spec_choice,
    x,

    phase,
    coupling,

    detuning,
    damping,
    pressure,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'phase vs coupling homodyne')

# PSDs $g$ vs. $\phi_2$

In [ ]:
spec_choice = 0
ani_params = (
    spec_choice,
    x,

    coupling,
    phase,

    damping,
    detuning,
    pressure,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'coupling vs phase x1 PSD')

In [ ]:
spec_choice = 4
ani_params = (
    spec_choice,
    x,

    coupling,
    phase,

    damping,
    detuning,
    pressure,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'coupling vs phase y2 PSD')

# Het. $g$ vs. $\phi_2$

In [ ]:
spec_choice = 'het'
ani_params = (
    spec_choice,
    x,

    coupling,
    phase,

    damping,
    detuning,
    pressure,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'coupling vs phase heterodyne')

# Hom. $g$ vs. $\phi_2$

In [ ]:
spec_choice = 'hom'
ani_params = (
    spec_choice,
    x,

    coupling,
    phase,

    detuning,
    damping,
    pressure,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'coupling vs phase homodyne')

# PSDs $\Delta$ vs. $g$

In [ ]:
spec_choice = 0
ani_params = (
    spec_choice,
    x,

    detuning,
    coupling,

    damping,
    phase,
    pressure,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'detuning vs coupling x1 PSD')

In [ ]:
spec_choice = 4
ani_params = (
    spec_choice,
    x,

    detuning,
    coupling,

    damping,
    phase,
    pressure,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'detuning vs coupling y2 PSD')

# Het. $\Delta$ vs. $g$

In [ ]:
spec_choice = 'het'
ani_params = (
    spec_choice,
    x,

    detuning,
    coupling,

    damping,
    phase,
    pressure,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'detuning vs coupling heterodyne')

# Hom. $\Delta$ vs. $g$

In [ ]:
spec_choice = 'hom'
ani_params = (
    spec_choice,
    x,

    detuning,
    coupling,

    damping,
    phase,
    pressure,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'detuning vs coupling homodyne')

# PSDs $g$ vs. $P_{air}$

In [ ]:
spec_choice = 0
ani_params = (
    spec_choice, x,
    coupling,
    pressure,
    damping,
    phase,
    detuning,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'coupling vs pressure x1 PSD')

In [ ]:
spec_choice = 4
ani_params = (
    spec_choice, x,
    coupling,
    pressure,
    damping,
    phase,
    detuning,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'coupling vs pressure y2 PSD')

# Het. $g$ vs. $P_{air}$

In [ ]:
spec_choice = 'het'
ani_params = (
    spec_choice, x,
    coupling,
    pressure,
    damping,
    phase,
    detuning,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'coupling vs pressure heterodyne')

# Hom. $g$ vs. $P_{air}$

In [ ]:
spec_choice = 'hom'
ani_params = (
    spec_choice, x,
    coupling,
    pressure,
    damping,
    phase,
    detuning,
)
ani = generate_heatmap_animation(*ani_params)

In [ ]:
if MAKE_ANIMATIONS:
    save_animation(ani, 'coupling vs pressure homodyne')